In [ ]:
#MLP vs CNN on MNIST using PyTorch

import torch #main pytorch library for tensor operations and deep learning
import torch.nn as nn #neural network module with layers and loss functions
import torch.optim as optim #for optimization algorithms such as SGD, Adam
from torchvision import datasets, transforms #for datasets and image transformation utilities
from torch.utils.data import DataLoader #for batching and loading data efficiently
import time #for speed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device) #using gpu or cpu

#a sequence of preprocessing steps to apply to every image
transform = transforms.Compose([
    transforms.ToTensor() '''image pixels into PyTorch tensor''' ,
    transforms.Normalize((0.1307,), (0.3081,)) '''normalizing the tensor as per mean and std deviation of MNIST dataset'''
])
#loading training data and applying preprocessing transformations
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)
#loading testing data
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

#MLP Model

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.model(x)

#CNN Model

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

#functions
#counting total number of trainable parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad) #summing the number of elements in all trainable tensors

#training the neural network and measuring time
def train(model, train_loader, criterion, optimizer, epochs=5):
    #set model to training mode
    model.train()
    #starting time of training
    start_time = time.time()

    #repeat training for specified number of epochs
    for epoch in range(epochs):

        total_loss = 0 #storing the loss for current epoch

        for images, labels in train_loader:
            #moving data to selected device
            images, labels = images.to(device), labels.to(device)

            #forward pass through the network
            outputs = model(images)
            #loss
            loss = criterion(outputs, labels)

            #clear gradients from prev iteration
            optimizer.zero_grad()

            #compute gradients using backprop
            loss.backward()

            #update model parameters using optimizer
            optimizer.step()

             #add current batch loss to running epoch loss
            total_loss += loss.item()
        #avg loss over all batches
        avg_loss = total_loss / len(train_loader)
        #displaying epoch number and loss
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

    end_time = time.time() #return end time of training
    return end_time - start_time #total duration of training

#evaluate the trained model on the test dataset and calculate accuracy
def evaluate(model, test_loader):
    # setting the model to evaluation mode
    model.eval()
    #count the number of correct predictions
    correct = 0
    #count the total number of test samples
    total = 0
    #disabling gradient computation to speed up evaluation
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            #predictions from the model
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            #correct predictions counting
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy

#training and evaluation of MLP
#MLP model to selected device
mlp = MLP().to(device)
#loss function for multi-class classification
criterion = nn.CrossEntropyLoss()
#adam optimizer to update mlp parameters
optimizer = optim.Adam(mlp.parameters(), lr=0.001)

print("\nTraining MLP...")
#counting the total number of trainable parameters
mlp_params = count_parameters(mlp)
#train the mlp model and record the training duration
mlp_time = train(mlp, train_loader, criterion, optimizer, epochs=5)
#evaluate the trained mlp model on the test dataset
mlp_accuracy = evaluate(mlp, test_loader)

#training and evaluation of CNN

cnn = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn.parameters(), lr=0.001)

print("\nTraining CNN...")
cnn_params = count_parameters(cnn)
cnn_time = train(cnn, train_loader, criterion, optimizer, epochs=5)
cnn_accuracy = evaluate(cnn, test_loader)

#FINAL COMPARISON

print("\nFinal Comparison")
print("-" * 50)
print(f"MLP Accuracy: {mlp_accuracy:.2f}%")
print(f"CNN Accuracy: {cnn_accuracy:.2f}%")
print(f"MLP Training Time: {mlp_time:.2f} seconds")
print(f"CNN Training Time: {cnn_time:.2f} seconds")
print(f"MLP Parameters: {mlp_params}")
print(f"CNN Parameters: {cnn_params}")

Using device: cpu


100%|██████████| 9.91M/9.91M [00:00<00:00, 63.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.74MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.9MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.18MB/s]



Training MLP...
Epoch [1/5], Loss: 0.2291
Epoch [2/5], Loss: 0.0930
Epoch [3/5], Loss: 0.0662
Epoch [4/5], Loss: 0.0505
Epoch [5/5], Loss: 0.0406

Training CNN...
Epoch [1/5], Loss: 0.1322
Epoch [2/5], Loss: 0.0442
Epoch [3/5], Loss: 0.0286
Epoch [4/5], Loss: 0.0220
Epoch [5/5], Loss: 0.0158

Final Comparison
--------------------------------------------------
MLP Accuracy: 97.48%
CNN Accuracy: 99.08%
MLP Training Time: 84.84 seconds
CNN Training Time: 394.53 seconds
MLP Parameters: 235146
CNN Parameters: 421642


The CNN achieved a higher classification accuracy of 99.08% compared to 97.48% for the MLP. This improvement is expected because convolutional layers are specifically designed to extract spatial features from images, allowing the CNN to recognize patterns such as edges, shapes, and digit structures more effectively than a fully connected network.

Both models achieved high classification performance on the MNIST dataset. The MLP provided faster training with fewer parameters, while the CNN delivered superior accuracy and lower loss. Therefore, although the CNN is computationally more expensive, it is the preferred model for image classification tasks where higher predictive performance is required.